# 02b — GPS, Patrones Temporales y Zonas Peligrosas
Extiende el análisis: estadísticas GPS del coche, patrones horarios, score de peligrosidad por km y validación de zonas conocidas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from pathlib import Path
from math import radians, cos, sin, asin, sqrt

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

CLASSES = ['Horn', 'Siren', 'Speech', 'Ringtone', 'Cry', 'Pets', 'Physiological', 'Vibration', 'Alarma']

data_path = Path('../data/processed')
pred_geo = pd.read_parquet(data_path / 'predictions_geo.parquet')
tracks   = pd.read_parquet(data_path / 'tracks.parquet')

if 'class_name' not in pred_geo.columns:
    pred_geo['class_name'] = pred_geo['class'].apply(lambda x: CLASSES[int(x)])

pred_geo['hour'] = pred_geo['t_start'].dt.tz_convert('Europe/Madrid').dt.hour

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 2*asin(sqrt(a))*6371  # km

print(f'Eventos geo: {len(pred_geo)} | Tracks: {len(tracks)} puntos GPS | Días: {tracks["date"].nunique()}')
Path("../outputs").mkdir(parents=True, exist_ok=True)


## 9. Estadísticas GPS del Coche
Velocidad punto a punto derivada de la diferencia temporal entre trackpoints consecutivos. Permite detectar paradas, frenadas bruscas y comparar tráfico entre días.

In [ ]:
# Calcular velocidad por punto GPS
tracks_sorted = tracks.sort_values(['date', 'direction', 'time']).copy()
tracks_sorted['dt_s'] = tracks_sorted.groupby(['date','direction'])['time'].diff().dt.total_seconds()

dists = []
prev_row = None
for _, row in tracks_sorted.iterrows():
    if prev_row is None or row['date'] != prev_row['date'] or row['direction'] != prev_row['direction']:
        dists.append(0.0)
    else:
        dists.append(haversine(prev_row['lon'], prev_row['lat'], row['lon'], row['lat']))
    prev_row = row
tracks_sorted['dist_km'] = dists

# Velocidad (km/h), filtrar outliers GPS (> 130 km/h o gap > 60s)
valid = (tracks_sorted['dt_s'] > 0) & (tracks_sorted['dt_s'] <= 60)
tracks_sorted.loc[valid, 'speed_kmh'] = (tracks_sorted.loc[valid,'dist_km'] / tracks_sorted.loc[valid,'dt_s']) * 3600
tracks_sorted['speed_kmh'] = tracks_sorted['speed_kmh'].clip(0, 130)
tracks_sorted['speed_smooth'] = tracks_sorted.groupby(['date','direction'])['speed_kmh'].transform(
    lambda s: s.rolling(7, min_periods=1, center=True).mean()
)

print('Velocidad media global:', round(tracks_sorted['speed_smooth'].mean(), 2), 'km/h')
print('Velocidad p95:', round(tracks_sorted['speed_smooth'].quantile(0.95), 2), 'km/h')

In [ ]:
# Distribución de velocidad por día
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot por día
speed_data = tracks_sorted.dropna(subset=['speed_smooth'])
days_sorted = sorted(speed_data['date'].unique())
ax = axes[0]
data_by_day = [speed_data[speed_data['date']==d]['speed_smooth'].values for d in days_sorted]
ax.boxplot(data_by_day, labels=[d[3:] for d in days_sorted], vert=True)
ax.set_title('Velocidad por Día (km/h)')
ax.set_xlabel('Día')
ax.set_ylabel('km/h')
ax.tick_params(axis='x', rotation=45)

# Distribución global
axes[1].hist(speed_data['speed_smooth'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(speed_data['speed_smooth'].mean(), color='red', linestyle='--', label='Media')
axes[1].axvline(50, color='orange', linestyle=':', label='50 km/h (urbano)')
axes[1].set_title('Distribución Global de Velocidad')
axes[1].set_xlabel('km/h')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/gps_velocity_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Tabla resumen de trayectos: duración, distancia, velocidad media
summary = tracks_sorted.groupby(['date','direction']).agg(
    distancia_km=('dist_km', 'sum'),
    duracion_min=('dt_s', lambda x: x.sum() / 60),
    vel_media=('speed_smooth', 'mean'),
    vel_max=('speed_smooth', 'max')
).round(2).reset_index()
summary['fecha'] = summary['date'].str[-4:] + '-' + summary['date'].str[3:5] + '-' + summary['date'].str[:2]
summary = summary.sort_values(['fecha','direction'])
display(summary[['date','direction','distancia_km','duracion_min','vel_media','vel_max']])

In [ ]:
# Detección de paradas (v < 5 km/h durante >= 10s) y frenadas bruscas (Dv > 20 km/h en 1 intervalo)
stops = tracks_sorted[(tracks_sorted['speed_smooth'] < 5) & (tracks_sorted['dt_s'] >= 10)].copy()
print(f'Paradas detectadas (v<5 km/h, dt>=10s): {len(stops)}')

tracks_sorted['delta_v'] = tracks_sorted.groupby(['date','direction'])['speed_smooth'].diff().abs()
hard_braking = tracks_sorted[tracks_sorted['delta_v'] > 20].copy()
print(f'Frenadas bruscas (|Δv| > 20 km/h): {len(hard_braking)}')

# Paradas por día
stop_by_day = stops.groupby('date').size().rename('paradas')
brake_by_day = hard_braking.groupby('date').size().rename('frenadas')
driving_stats = pd.concat([stop_by_day, brake_by_day], axis=1).fillna(0).astype(int)
print('\nParadas y Frenadas bruscas por día:')
display(driving_stats)

## 10. Patrones Temporales y Duración de Eventos
Distribución de detecciones por hora del día (rush hour vs fuera de pico) y duración característica por clase.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución por hora del día
hour_counts = pred_geo.groupby('hour').size()
all_hours = pd.Series(0, index=range(24))
hour_counts = all_hours.add(hour_counts, fill_value=0).astype(int)
axes[0].bar(hour_counts.index, hour_counts.values, color='steelblue', edgecolor='white')
axes[0].axvspan(7, 9.5, alpha=0.15, color='orange', label='Rush mañana (7-9h)')
axes[0].axvspan(17, 19.5, alpha=0.15, color='red', label='Rush tarde (17-19h)')
axes[0].set_title('Detecciones por Hora del Día')
axes[0].set_xlabel('Hora (hora local)')
axes[0].set_ylabel('Nº detecciones')
axes[0].legend()

# Distribución por hora y clase (solo Horn y Siren)
danger = pred_geo[pred_geo['class_name'].isin(['Horn','Siren'])]
for cls, grp in danger.groupby('class_name'):
    axes[1].hist(grp['hour'], bins=24, range=(0,24), alpha=0.6, label=cls)
axes[1].set_title('Horn y Siren por Hora')
axes[1].set_xlabel('Hora (hora local)')
axes[1].set_ylabel('Nº detecciones')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/temporal_patterns.png', bbox_inches='tight')
plt.show()

In [ ]:
# Duración media por clase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

duration_data = pred_geo.dropna(subset=['duration_s'])
sns.boxplot(data=duration_data, x='class_name', y='duration_s', ax=axes[0],
            palette='Set2', hue='class_name', legend=False, order=CLASSES)
axes[0].set_title('Duración por Clase (segundos)')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Duración (s)')
axes[0].tick_params(axis='x', rotation=45)

# Media de duración por clase
dur_mean = duration_data.groupby('class_name')['duration_s'].mean().reindex(CLASSES)
axes[1].barh(CLASSES, dur_mean.values, color='coral')
axes[1].set_title('Duración Media por Clase')
axes[1].set_xlabel('Duración media (s)')
for i, v in enumerate(dur_mean.values):
    if not np.isnan(v):
        axes[1].text(v + 0.1, i, f'{v:.2f}s', va='center')

plt.tight_layout()
plt.savefig('../outputs/duration_by_class.png', bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de co-ocurrencia con numpy (eficiente: O(n*k))
WINDOW_S = 5
pred_sorted = pred_geo.sort_values("t_start").reset_index(drop=True)
t_ns = pred_sorted["t_start"].values.astype("int64")
cls_arr = pred_sorted["class"].fillna(0).astype(int).values
W = int(WINDOW_S * 1e9)

cooc = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for i in range(len(t_ns)):
    lo = np.searchsorted(t_ns, t_ns[i] - W, side="left")
    hi = np.searchsorted(t_ns, t_ns[i] + W, side="right")
    for j in range(lo, hi):
        ci, cj = cls_arr[i], cls_arr[j]
        if ci != cj and ci < len(CLASSES) and cj < len(CLASSES):
            cooc[ci, cj] += 1

df_cooc = pd.DataFrame(cooc, index=CLASSES, columns=CLASSES)
plt.figure(figsize=(10, 8))
sns.heatmap(df_cooc, annot=True, fmt="d", cmap="YlOrRd",
            cbar_kws={"label": f"Co-ocurrencias (ventana ±{WINDOW_S}s)"})
plt.title(f"Matriz de Co-ocurrencia de Clases (ventana ±{WINDOW_S}s)")
plt.tight_layout()
plt.savefig("../outputs/cooccurrence_matrix.png", bbox_inches="tight")
plt.show()


## 11. Score de Peligrosidad por Segmento km
Horn y Siren son los indicadores más directos de conflicto vial. Calculamos densidad por km y un score compuesto para rankear segmentos.

In [ ]:
# Unir pred_geo con cum_dist de tracks via merge_asof temporal
groups = []
for date_val, grp in tracks.groupby('date'):
    grp = grp.sort_values('time').copy()
    coords = grp[['lat','lon']].values
    dists = [0.0]
    for i in range(1, len(coords)):
        dists.append(haversine(coords[i-1][1], coords[i-1][0], coords[i][1], coords[i][0]))
    grp['cum_dist'] = np.cumsum(dists)
    groups.append(grp)
tracks_with_dist = pd.concat(groups, ignore_index=True)

events = pred_geo.sort_values('t_start').copy()
events['date_str'] = events['t_start'].dt.tz_convert('UTC').dt.strftime('%d-%m-%Y')
tracks_s = tracks_with_dist.sort_values('time').copy()
events = pd.merge_asof(events, tracks_s[['time','cum_dist','date']],
                       left_on='t_start', right_on='time', by='date', direction='nearest')
events['segment'] = (events['cum_dist'] // 1.0).astype('Int64')
events_valid = events.dropna(subset=['segment'])
print(f'Eventos con segmento asignado: {len(events_valid)} / {len(events)}')

In [ ]:
# Densidad Horn y Siren por km
max_seg = int(events_valid['segment'].max())
horn_density   = events_valid[events_valid['class_name']=='Horn'].groupby('segment').size()
siren_density  = events_valid[events_valid['class_name']=='Siren'].groupby('segment').size()
total_density  = events_valid.groupby('segment').size()

seg_idx = pd.RangeIndex(max_seg + 1)
df_danger = pd.DataFrame({
    'horn':  horn_density.reindex(seg_idx, fill_value=0),
    'siren': siren_density.reindex(seg_idx, fill_value=0),
    'total': total_density.reindex(seg_idx, fill_value=0),
})

# Score compuesto: Siren pesa el doble (emergencia > bocina)
df_danger['danger_score'] = df_danger['horn'] * 1.0 + df_danger['siren'] * 2.0

# Normalizar a 0-100
mx = df_danger['danger_score'].max()
df_danger['danger_score_norm'] = (df_danger['danger_score'] / mx * 100).round(1) if mx > 0 else 0

print('Top 10 segmentos más peligrosos:')
display(df_danger.sort_values('danger_score', ascending=False).head(10))

In [ ]:
# Visualización: Horn, Siren y score por km
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].bar(df_danger.index, df_danger['horn'], color='orange', label='Horn')
axes[0].set_ylabel('Eventos Horn')
axes[0].set_title('Densidad Horn por Km')

axes[1].bar(df_danger.index, df_danger['siren'], color='red', label='Siren')
axes[1].set_ylabel('Eventos Siren')
axes[1].set_title('Densidad Siren por Km')

colors = ['green' if s < 25 else 'yellow' if s < 50 else 'orange' if s < 75 else 'red'
          for s in df_danger['danger_score_norm']]
axes[2].bar(df_danger.index, df_danger['danger_score_norm'], color=colors)
axes[2].set_ylabel('Score (0-100)')
axes[2].set_title('Score de Peligrosidad Compuesto (Horn + 2×Siren, normalizado)')
axes[2].set_xlabel('Segmento (km)')

for ax in axes:
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/danger_score_by_km.png', bbox_inches='tight')
plt.show()

# Guardar para uso en NB03
df_danger.to_parquet(data_path / 'danger_scores.parquet')
print('danger_scores.parquet guardado.')

## 12. Comparativa Ida vs Vuelta (Mann-Whitney U)
¿Hay diferencias significativas en la distribución de eventos entre la dirección de ida y vuelta?

In [ ]:
# Unir dirección de trayecto a eventos via merge_asof
events_dir = pd.merge_asof(events.sort_values('t_start'),
                           tracks_with_dist.sort_values('time')[['time','direction','date']],
                           left_on='t_start', right_on='time', by='date', direction='nearest')

results = []
for cls in CLASSES:
    ida    = events_dir[(events_dir['class_name']==cls) & (events_dir['direction']=='ida')]['confidence']
    vuelta = events_dir[(events_dir['class_name']==cls) & (events_dir['direction']=='vuelta')]['confidence']
    if len(ida) < 5 or len(vuelta) < 5:
        results.append({'clase': cls, 'n_ida': len(ida), 'n_vuelta': len(vuelta), 'p_value': np.nan, 'significativo': '-'})
        continue
    stat, p = sp_stats.mannwhitneyu(ida, vuelta, alternative='two-sided')
    results.append({'clase': cls, 'n_ida': len(ida), 'n_vuelta': len(vuelta),
                    'p_value': round(p, 4), 'significativo': 'SI (p<0.05)' if p < 0.05 else 'No'})

df_mw = pd.DataFrame(results)
print('Test Mann-Whitney U: confianza ida vs vuelta por clase')
display(df_mw)

In [ ]:
# Conteo de eventos por dirección y clase
dir_counts = events_dir.groupby(['direction','class_name']).size().unstack(fill_value=0)
dir_counts = dir_counts.reindex(columns=CLASSES, fill_value=0)

dir_counts.plot(kind='bar', figsize=(12, 5), colormap='tab20')
plt.title('Distribución de Clases por Dirección del Trayecto')
plt.xlabel('Dirección')
plt.ylabel('Detecciones')
plt.xticks(rotation=0)
plt.legend(title='Clase', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/class_by_direction.png', bbox_inches='tight')
plt.show()

## 13. Validación de Zonas Peligrosas Conocidas
Comparar el score calculado con las 2 zonas que se sabe que son peligrosas. Editar las coordenadas y nombres en la celda de configuración.

In [ ]:
# ⚠️ CONFIGURACIÓN: editar con las coordenadas reales de las zonas conocidas
KNOWN_ZONES = [
    {'nombre': 'Zona Peligrosa 1',  'lat': 39.47, 'lon': -0.37, 'radio_m': 500},
    {'nombre': 'Zona Peligrosa 2',  'lat': 39.48, 'lon': -0.39, 'radio_m': 500},
]

def dist_m(lat1, lon1, lat2, lon2):
    return haversine(lon1, lat1, lon2, lat2) * 1000

results_z = []
for zone in KNOWN_ZONES:
    mask = events_valid.apply(
        lambda r: dist_m(r['lat'], r['lon'], zone['lat'], zone['lon']) <= zone['radio_m']
        if pd.notna(r['lat']) else False, axis=1
    )
    subset = events_valid[mask]
    horn_n  = (subset['class_name'] == 'Horn').sum()
    siren_n = (subset['class_name'] == 'Siren').sum()
    score_z = horn_n * 1.0 + siren_n * 2.0
    results_z.append({
        'zona': zone['nombre'],
        'radio_m': zone['radio_m'],
        'total_eventos': len(subset),
        'horn': horn_n,
        'siren': siren_n,
        'danger_score': score_z,
    })

df_zones = pd.DataFrame(results_z)
display(df_zones)

In [ ]:
# Comparar score de zonas conocidas vs percentiles globales
p50 = df_danger['danger_score'].median()
p75 = df_danger['danger_score'].quantile(0.75)
p90 = df_danger['danger_score'].quantile(0.90)

print(f'Percentiles globales de danger_score:')
print(f'  p50 = {p50:.1f} | p75 = {p75:.1f} | p90 = {p90:.1f}')
print()

for _, z in df_zones.iterrows():
    s = z['danger_score']
    if s >= p90:
        nivel = 'TOP 10% — zona confirmada como MUY peligrosa'
    elif s >= p75:
        nivel = 'TOP 25% — zona peligrosa'
    elif s >= p50:
        nivel = 'Por encima de la mediana'
    else:
        nivel = 'Por debajo de la mediana — revisar coordenadas'
    print(f'{z["zona"]}: score={s:.1f} → {nivel}')